### Script para atualizar a tabela de historico de pagamento retirada do SAP e agrupando ela ao considado em uma Delta ###

In [0]:
#Carrega o normalizador/importador de Bibliotecas
%run "/Workspace/Jurídico/Normalizador_de_dados/Normalizador"  

In [0]:
# Carrega a base de pagamento historica 

base_pagamentos = f'/Volumes/databox/juridico_comum/arquivos/financeiro/Br_Consolidado_pagamentos_honorarios/Base Pagamentos Jurídico..xlsx'

In [0]:
df_pagamentos = pd.read_excel(
    base_pagamentos,
    sheet_name='VIA VAREJO',
    header=1,
    engine='openpyxl'
)

In [0]:
#dropa coluna vazia usada para separar viasualmente
df_pagamentos.drop(columns=['Unnamed: 35'], inplace=True)

In [0]:
df_pagamentos.columns

In [0]:
import unicodedata

def normalizador(texto):
    #Retira os acentos e remove espaços
    nks = unicodedata.normalize('NFD', texto)
    sem_acentos = "".join([c for c in nks if not unicodedata.combining(c)])
    # Remove espaços e coloca em minúsculo
    return sem_acentos.replace(" ", "_").upper()
# Aplicando a todas as colunas do seu DataFrame
df_pagamentos.columns = [normalizador(col) for col in df_pagamentos.columns]

In [0]:
df_pagamentos.head()

In [0]:
df_pagamentos.dtypes

In [0]:
df_pagamentos.head()

In [0]:
# 1. Pega as colunas object
cols_object = df_pagamentos.select_dtypes(include=['object']).columns

# 2. Itera uma por uma (evita o erro de length)
for col in cols_object:
    # Converte para string e substitui as variações de nulo por None
    df_pagamentos[col] = df_pagamentos[col].astype(str).replace(
        ['nan', 'NaN', '', 'None', 'NAT'], None
    )

In [0]:
#Transforma o df para spark
df_spark = spark.createDataFrame(df_pagamentos)

df_spark = df_spark.withColumnRenamed('CHAVE_(DATA_DA_MIGO_+_NOTA_FISCAL)','CHAVE_DATA_DA_MIGO_E_NOTA_FISCAL') \
                   .withColumnRenamed('CHAVE_(AREA_+_REMUNERACAO)','CHAVE_AREA_REMUNERACAO') \
                    .withColumnRenamed('CHAVE_(ESCRITORIO_+_NUMERO_NOTA)','CHAVE_ESCRITORIO_NUMERO_NOTA') \
                    .withColumnRenamed('N°REQUISICAO','N_REQUISICAO')
df_spark.display()

In [0]:
#Salva em uma Delta
df_spark.write.format('delta').mode('overwrite').option('overwriteSchema', 'true').saveAsTable('databox.juridico_comum.br_Consolidado_pagamentos_honorarios')

print("Sucesso!")